**Notice**<br>
‼️ Dengan ini saya menyatakan bahwa soal ini saya kerjakan secara mandiri

> *adapted from: Longa*

**Scope:** This notebook completes a Cora graph data-processing exercise and a pointwise Conv1d demonstration. The Conv1d layer does not use graph edges or perform message passing; it is not a trained GNN classifier. Original exercise credits: “adapted from: Longa”.


# Environment setup

## Seed - reproducibility

In [ ]:
# Install the packages in requirements.txt before opening this notebook.


In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch

np.random.seed(91)
torch.manual_seed(91)

In [ ]:
print(torch.__version__)

use_CUDA = torch.cuda.is_available()
use_MPS = torch.backends.mps.is_available()

print(f"CUDA: {use_CUDA}")
print(f"MPS: {use_MPS}")

# Dataset download

In [ ]:
# The Cora dataset is downloaded by Planetoid on first run.


In [ ]:
from torch_geometric.datasets import Planetoid

In [ ]:
from pathlib import Path
dataset_ = Planetoid(root=str(Path('data/Cora')), name='Cora')


## Melihat properti dataset

In [ ]:
print(dataset_)

In [ ]:
print(f"Jumlah graf: {len(dataset_)}")

In [ ]:
print(f'Jumlah kelas (label): {dataset_.num_classes}')
NC = torch.unique(dataset_[0].y).numel()
print(NC)
assert NC == dataset_.num_classes
print(f'Jumlah fitur node: {dataset_.num_node_features}')
FN = dataset_[0].x.size(1)
print(FN)
assert FN == dataset_.num_node_features


## Melihat atribut graf

In [ ]:
print(f'edge_index:\n{dataset_[0].edge_index}')


In [ ]:
print(f'train mask:\n{dataset_[0].train_mask}')


## Ambil Graf Satu-satunya

In [ ]:
data = dataset_[0]
print(f"Data:\r\n{data}")

# Memproses Data Graf (Subgraf, Hops, dst.)

In [ ]:
from torch_geometric.utils import subgraph, degree, coalesce, k_hop_subgraph, to_networkx
from torch_geometric.data import Data

In [ ]:
nsrc = data.edge_index[0]
nsrc


In [ ]:
assert type(nsrc) == torch.Tensor

In [ ]:
# Mendapatkan Node degrees
npdeg = degree(nsrc, dtype=torch.long).numpy()
npdeg

In [ ]:
subset_indices = np.where(npdeg > 10)[0]
subset_indices


In [ ]:
assert type(subset_indices) == np.ndarray

In [ ]:
# Select up to 50 unique high-degree node IDs.
subset_indices = torch.as_tensor(subset_indices[:50], dtype=torch.long)
subset_indices


In [ ]:
ei, ea = subgraph(subset_indices, data.edge_index)
print(f"ei:\r\n{ei}\r\n"
      f"coalesced:\r\n{coalesce(ei)}")
ei = coalesce(ei)

In [ ]:
ei.size()

In [ ]:
def plot_graph(G):
    layout = nx.spring_layout(G, k=1.5)
    nx.draw(G, layout, with_labels=True, node_color='lightblue', edge_color='gray', node_size=300, font_weight='bold')
    plt.show()

## Melabeli Ulang Node dengan SKLearn Label Encoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
a = ei.reshape(-1).cpu().numpy()
a


Melakukan proses fitting LabelEncoder pada data `a`

In [ ]:
le = LabelEncoder()
le.fit(a)

Membangun edge_index dari 2 vector yang telah ter-transform,<br>
Satu sebagai asal (`src`) dan tujuan (`dst`)

In [ ]:
src_ = le.transform(ei[0])
dst_ = le.transform(ei[1])

print(f"src:\r\n{src_}\r\ndst:\r\n{dst_}")

In [ ]:
ei_t = np.vstack([src_, dst_])
ei_t = torch.tensor(ei_t)
ei_t

In [ ]:
ei_t.size()

In [ ]:
assert type(ei_t) == type(ei)

In [ ]:
subgraf_t = Data(edge_index=ei_t, num_nodes=int(ei_t.max().item()) + 1)
subgraf_t


In [ ]:
subgraf_t.edge_index

## Melakukan plot graf

In [ ]:
G_t = to_networkx(subgraf_t)
plot_graph(G_t)

# Memproses K-Hop Subgraf

In [ ]:
subset, ei, mapping, em = k_hop_subgraph(
    node_idx=9, num_hops=2, edge_index=data.edge_index,
    relabel_nodes=True, num_nodes=data.num_nodes
)


In [ ]:
subset

In [ ]:
ei = coalesce(ei)
ei

In [ ]:
xnew = data.x[subset, 72:75]
xnew


In [ ]:
xnew.unsqueeze(-1).shape
xnew

In [ ]:
xnew.shape

## Membuat dataset dari subgraf

In [ ]:
dsub = Data(
    x=xnew, y=data.y[subset], edge_index=ei,
    train_mask=data.train_mask[subset],
    val_mask=data.val_mask[subset],
    test_mask=data.test_mask[subset],
    num_nodes=len(subset),
)
dsub


In [ ]:
print(f"x: {dsub.x.shape}, ei: {ei.shape}")
print(f"train_mask:\r\n{dsub.train_mask}\r\nval_mask:\r\n{dsub.val_mask}")

# Conv Net

In [ ]:
import torch.nn.functional as F
from torch.nn import Conv1d

In [ ]:
class GNNConv1D(torch.nn.Module):
    def __init__(self, inc, outc):
        super().__init__()
        self.conv1 = Conv1d(inc, outc, kernel_size=1)
        with torch.no_grad():
            self.conv1.weight.fill_(0.5)
            self.conv1.bias.fill_(0.1)

    def forward(self, inp_):
        # This is a pointwise feature transform; edge_index is not used.
        x = inp_.x.unsqueeze(-1)
        return self.conv1(x).squeeze(-1)


In [ ]:
dev = torch.device('cuda' if torch.cuda.is_available() else
                   'mps' if torch.backends.mps.is_available() else 'cpu')
print(dev)


In [ ]:
mod_ = GNNConv1D(3, dataset_.num_classes).to(dev)
print(mod_)

inp = dsub.to(dev)

In [ ]:
rv = mod_(inp)
rv

In [ ]:
label_ = torch.ones(dataset_.num_classes)

In [ ]:
loss_fn = torch.nn.MSELoss()
label_ = label_.to(dev)
ls_ = loss_fn(rv[4], label_)
print(f'Loss: {ls_.item():.4f}')


In [ ]:
bw = ls_.backward(retain_graph=True)
bw

In [ ]:
# Mencatak nilai backpropagation loss
ls_.item()